## Deployments Deep Dive

Load API Key

In [ ]:
# Option 1: Load from .env file
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Get API key from environment variables
ENKRYPTAI_API_KEY = os.getenv("ENKRYPTAI_API_KEY")

# Option 2: Paste your API key directly (not recommended for production)
# ENKRYPTAI_API_KEY = "your_api_key_here"

#print the api key
print(ENKRYPTAI_API_KEY)


In [ ]:
def send_enkrypt_request(message="Hello", deployment="mortgage deploy 0"):
    """
    Send a request to the EnkryptAI API.
    
    Args:
        message (str): The message to send to the API
        deployment (str): The deployment name to use
        
    Returns:
        dict: The JSON response from the API
    """
    import requests
    
    url = "https://api.enkryptai.com/ai-proxy/chat/completions"
    
    headers = {
        "Content-Type": "application/json",
        "X-Enkrypt-Deployment": deployment, # GR -> ENDPOINT (model and provider) -> GR -> Back to User
        "apikey": ENKRYPTAI_API_KEY
    }
    
    payload = {
        "model": "gpt-4o", # Unneccessarily specify model again? User can choose any model in the provider of the endpoint.
        "messages": [
            {
                "role": "user",
                "content": message
            }
        ]
    }

    # Text fromn the user -> content from the user message -> LAST USER MESSAGE.
    # Text proxies to the inpuit GR through kong. If we detect an attack, we abort and return the detected response.
    # If not, we proxy directing to the endpoint from the provider (openai, together, anthropic, etc)
    # -- You will get the model name from the endpoint details. (also get system prompt from the endpoint details)
    # We get response, and then check output GR through kong. If we detect an attack, we just append the detection response.

    # Might be a breaking change. If we dont need the model, we should not be accepting the model parameter anymore.
 
    
    response = requests.post(url, headers=headers, json=payload)
    return response.json()

# Example usage
json_response = send_enkrypt_request()
print(json_response)

In [ ]:
import json

# Pretty print the JSON response with indentation
print(json.dumps(json_response, indent=4))


In [ ]:
def process_enkrypt_response(json_response):
    # Check input guardrails
    input_summary = json_response.get('enkrypt_policy_detections', {}).get('input_guardrails', {}).get('summary', {})
    input_details = json_response.get('enkrypt_policy_detections', {}).get('input_guardrails', {}).get('details', {})
    if any(value == 1 for value in input_summary.values()):

        body  = {
            "message": "Violation detected in input prompt",
            "details": input_details
        }

        return body
    
    # Check output guardrails
    output_summary = json_response.get('enkrypt_policy_detections', {}).get('output_guardrails', {}).get('summary', {})
    output_details = json_response.get('enkrypt_policy_detections', {}).get('output_guardrails', {}).get('details', {})
    if any(value == 1 for value in output_summary.values()):
        body  = {
            "message": "Violation detected in output response",
            "details": output_details
        }
        
        return body
    
    # If no violations, return the message content

    content = json_response.get('choices', [{}])[0].get('message', {}).get('content', 'No content found')

    body = {
        "message": content
    }

    return body

# Example usage:
result = process_enkrypt_response(json_response)
print(result)


Now together in an application

In [ ]:
prompt = "Tell me about closing costs?"

json_response = send_enkrypt_request(prompt, "home loan chatbot")

result = process_enkrypt_response(json_response)

print(json.dumps(result, indent=4))
